In [1]:
import subprocess
import sys

subprocess.check_call([
    "uv",
    "pip",
    "install",
    "--python",
    sys.executable,
    "transformers<5",
    "tokenizers<0.21",
    "datasets",
    "accelerate",
    "peft",
    "sentencepiece",
    "protobuf<5",
])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "jupyter"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "ipywidgets"])

Resolved 71 packages in 370ms
 Downloaded sentencepiece
 Downloaded aiohttp
 Downloaded tokenizers
 Downloaded pyarrow
Prepared 10 packages in 734ms
Uninstalled 3 packages in 58ms
Installed 19 packages in 15ms
 + accelerate==1.13.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + datasets==4.8.4
 + dill==0.4.1
 + frozenlist==1.8.0
 - fsspec==2026.3.0
 + fsspec==2026.2.0
 + multidict==6.7.1
 + multiprocess==0.70.19
 + peft==0.19.1
 + propcache==0.4.1
 + protobuf==4.25.9
 + pyarrow==23.0.1
 + sentencepiece==0.2.1
 - tokenizers==0.22.2
 + tokenizers==0.20.3
 - transformers==4.57.6
 + transformers==4.46.3
 + xxhash==3.6.0
 + yarl==1.23.0


0

## Many libraries

In [2]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import  AutoTokenizer, AutoModelForSequenceClassification,  Trainer, TrainingArguments,  EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import random
import os
import pandas as pd
import sys
import tempfile
import time
from functools import partial
from pathlib import Path
import json
from collections import Counter

/home/sortmon/UPV_EARTH_PROYECTOIII/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
DATASET_CANDIDATES = [
    Path("EXIST2026_training.json"),
    Path("data/EXIST2026_training.json"),
    Path("../pract_4/EXIST2026_training.json"),
]

dataset_path = next((str(p) for p in DATASET_CANDIDATES if p.exists()), str(DATASET_CANDIDATES[-1]))
print(f"Using dataset path: {dataset_path}")

'../pract_4/EXIST2026_training.json'

In [4]:
DATASET_FILE = "EXIST2026_training"


LABELS_21 = ['NO', 'YES']
LABELS_22 = ['JUDGEMENTAL', 'DIRECT']
LABELS_23 = ['IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']

MAPPING_21 = {"YES": 1, "NO": 0}
MAPPING_22 = {"DIRECT": 1, "JUDGEMENTAL": 0}

DEV_PART = 0.2

DATASET_PATH = Path(dataset_path)

## Read dataset

In [5]:
## Read and split the dataset for each task. Use the same code developed in the previous lab session - Lab_S4

def load_exist_dataset(dataset_file):
        try:
            with open(dataset_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            print(f"Loaded dataset from: {dataset_file}")
            return data
        except json.JSONDecodeError:
            raise FileNotFoundError(
        "No valid dataset JSON found."
    )
        

def majority_label(labels, valid_labels, threshold=4):
    """
    Return the majority label if it appears at least `threshold` times.
    Otherwise return None.
    """
    counts = Counter(label for label in labels if label in valid_labels)
    if not counts:
        return None

    label, count = counts.most_common(1)[0]
    return label if count >= threshold else None


def aggregate_multilabel(annotation_lists, valid_labels):
    """
    Aggregate subtask 2.3 labels by taking the union of all valid facet
    labels assigned by the annotators, discarding placeholders such as
    '-' or 'UNKNOWN'.

    This is consistent with the appendix: the hard-hard restriction is
    enforced for subtasks 2.1 and 2.2, whereas 2.3 uses positive memes
    and keeps their valid facet information.
    """
    valid_set = set(valid_labels)
    facets = set()

    for ann in annotation_lists:
        if isinstance(ann, str):
            ann = [ann]
        for label in ann:
            if label in valid_set:
                facets.add(label)

    return sorted(facets)


alljs = load_exist_dataset(DATASET_PATH)

rows = []
for item in alljs.values():
    label_21 = majority_label(item.get("labels_task2_1", []), LABELS_21)
    label_22 = majority_label(item.get("labels_task2_2", []), LABELS_22)
    label_23 = aggregate_multilabel(item.get("labels_task2_3", []), LABELS_23)

    rows.append(
        {
            "id_EXIST": item.get("id_EXIST"),
            "lang": item.get("lang"),
            "text": item.get("text", ""),
            "label_21": label_21,
            "label_22": label_22,
            "labels_23": label_23,
        }
    )

df = pd.DataFrame(rows)


def build_task_partitions(df, lang_code):
    """
    Build the three subtasks for one language.
    """
    df_lang = df[df["lang"] == lang_code].copy()

    # Subtask 2.1: only majority labels
    df_21 = df_lang[df_lang["label_21"].notna()].copy()
    df_21["label"] = df_21["label_21"]

    # Subtask 2.2: ONLY positive memes with a majority DIRECT/JUDGEMENTAL label
    df_22 = df_lang[
        (df_lang["label_21"] == "YES") &
        (df_lang["label_22"].notna())
    ].copy()
    df_22["label"] = df_22["label_22"]

    # Subtask 2.3: only positive memes with at least one valid facet label
    df_23 = df_lang[
        (df_lang["label_21"] == "YES") &
        (df_lang["labels_23"].map(len) > 0)
    ].copy()

    return df_21, df_22, df_23


def split_binary(df_task, label_col):
    """
    Stratified train/dev split for binary classification subtasks.
    """
    if len(df_task) == 0:
        return df_task.copy(), df_task.copy()
    if len(df_task) < 2:
        return (
            df_task.reset_index(drop=True),
            df_task.iloc[0:0].copy().reset_index(drop=True),
        )

    label_counts = df_task[label_col].value_counts()
    can_stratify = label_counts.min() >= 2 and label_counts.shape[0] > 1

    train_df, dev_df = train_test_split(
        df_task,
        test_size=DEV_PART,
        random_state=42,
        stratify=df_task[label_col] if can_stratify else None,
    )

    return train_df.reset_index(drop=True), dev_df.reset_index(drop=True)


def multilabel_split_score(train_labels, dev_labels, full_labels):
    """
    Lower is better. It combines:
    - absolute deviation in per-label prevalence
    - absolute deviation in mean label cardinality
    """
    full_prev = full_labels.mean(axis=0)
    train_prev = train_labels.mean(axis=0)
    dev_prev = dev_labels.mean(axis=0)

    prev_error = np.abs(train_prev - full_prev).mean() + np.abs(dev_prev - full_prev).mean()

    full_card = full_labels.sum(axis=1).mean()
    train_card = train_labels.sum(axis=1).mean()
    dev_card = dev_labels.sum(axis=1).mean()

    card_error = abs(train_card - full_card) + abs(dev_card - full_card)
    return float(prev_error + 0.25 * card_error)


def split_multilabel(df_task, n_trials=50):
    """
    Heuristic multilabel split:
    try several random splits and keep the one that best preserves
    label prevalence and average label cardinality.
    """
    if len(df_task) == 0:
        return df_task.copy(), df_task.copy()
    if len(df_task) < 2:
        return (
            df_task.reset_index(drop=True),
            df_task.iloc[0:0].copy().reset_index(drop=True),
        )

    label_matrix = pd.DataFrame(
        [[label in labs for label in LABELS_23] for labs in df_task["labels_23"]],
        columns=LABELS_23,
    ).astype(int).to_numpy()

    best_score = np.inf
    best_split = None

    for seed in range(42, 42 + n_trials):
        train_idx, dev_idx = train_test_split(
            np.arange(len(df_task)),
            test_size=DEV_PART,
            random_state=seed,
            shuffle=True,
        )

        train_labels = label_matrix[train_idx]
        dev_labels = label_matrix[dev_idx]
        score = multilabel_split_score(train_labels, dev_labels, label_matrix)

        if score < best_score:
            best_score = score
            best_split = (train_idx, dev_idx)

    train_idx, dev_idx = best_split
    train_df = df_task.iloc[train_idx].reset_index(drop=True)
    dev_df = df_task.iloc[dev_idx].reset_index(drop=True)
    return train_df, dev_df


def print_language_summary(lang_name, tr21, dv21, tr22, dv22, tr23, dv23):
    print(lang_name.upper())
    print("=" * len(lang_name))
    print()

    print("Subtask 2.1")
    total_21 = len(tr21) + len(dv21)
    yes_21 = (tr21["label_21"] == "YES").sum() + (dv21["label_21"] == "YES").sum()
    no_21 = (tr21["label_21"] == "NO").sum() + (dv21["label_21"] == "NO").sum()
    print(f"\tNumber of samples with majority label in subtask 2.1: {len(tr21)}/{len(dv21)}={total_21}")
    print(f"\t\tNumber of samples with 'YES' label in subtask 2.1: {(tr21['label_21'] == 'YES').sum()}/{(dv21['label_21'] == 'YES').sum()}={yes_21}")
    print(f"\t\tNumber of samples with 'NO' label in subtask 2.1: {(tr21['label_21'] == 'NO').sum()}/{(dv21['label_21'] == 'NO').sum()}={no_21}")

    print("Subtask 2.2")
    total_22 = len(tr22) + len(dv22)
    direct_22 = (tr22["label_22"] == "DIRECT").sum() + (dv22["label_22"] == "DIRECT").sum()
    judgmental_22 = (tr22["label_22"] == "JUDGEMENTAL").sum() + (dv22["label_22"] == "JUDGEMENTAL").sum()
    print(f"\tNumber of positive samples with majority label in subtask 2.2: {len(tr22)}/{len(dv22)}={total_22}")
    print(f"\t\tNumber of samples with 'DIRECT' label in subtask 2.2: {(tr22['label_22'] == 'DIRECT').sum()}/{(dv22['label_22'] == 'DIRECT').sum()}={direct_22}")
    print(f"\t\tNumber of samples with 'JUDGEMENTAL' label in subtask 2.2: {(tr22['label_22'] == 'JUDGEMENTAL').sum()}/{(dv22['label_22'] == 'JUDGEMENTAL').sum()}={judgmental_22}")

    print("Subtask 2.3")
    total_23 = len(tr23) + len(dv23)
    print(f"\tNumber of positive samples used in subtask 2.3: {len(tr23)}/{len(dv23)}={total_23}")
    for label in LABELS_23:
        train_count = tr23["labels_23"].apply(lambda labs: label in labs).sum()
        dev_count = dv23["labels_23"].apply(lambda labs: label in labs).sum()
        print(f"\t\t{label}: {train_count}/{dev_count}={train_count + dev_count}")

    print()


# Build language-specific datasets
en_21, en_22, en_23 = build_task_partitions(df, "en")
es_21, es_22, es_23 = build_task_partitions(df, "es")

# Train/dev splits
train21_en, dev21_en = split_binary(en_21, "label")
train22_en, dev22_en = split_binary(en_22, "label")
train23_en, dev23_en = split_multilabel(en_23)

train21_es, dev21_es = split_binary(es_21, "label")
train22_es, dev22_es = split_binary(es_22, "label")
train23_es, dev23_es = split_multilabel(es_23)

print_language_summary("English", train21_en, dev21_en, train22_en, dev22_en, train23_en, dev23_en)
print_language_summary("Spanish", train21_es, dev21_es, train22_es, dev22_es, train23_es, dev23_es)

FileNotFoundError: [Errno 2] No such file or directory: '../pract_4/EXIST2026_training.json'

In [ ]:
train22_en

,id_EXIST,lang,text,label_21,label_22,labels_23,label
0,211857,en,The Girl you like telling that it's her first ...,YES,DIRECT,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...",DIRECT
1,210167,en,"I DONT COOK, I DON'T CLEAN, I DON'T WANT KIDS,...",YES,JUDGEMENTAL,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, STER...",JUDGEMENTAL
2,211915,en,Women: Why can men go out shirtless but we can...,YES,DIRECT,"[IDEOLOGICAL-INEQUALITY, OBJECTIFICATION, SEXU...",DIRECT
3,211430,en,Redditor makes meme about female: Feminists in...,YES,DIRECT,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...",DIRECT
4,211617,en,I CANNOT COMMENT ON YOUR STEPMOM COWS ARE SACR...,YES,DIRECT,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...",DIRECT
...,...,...,...,...,...,...,...
346,211888,en,Women written by men Women written by women,YES,JUDGEMENTAL,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...",JUDGEMENTAL
347,210586,en,OPPRESSION LIBERATION... DADDY ISSU,YES,DIRECT,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...",DIRECT
348,211897,en,Third Wave Feminism Corporate needs you to fin...,YES,DIRECT,"[IDEOLOGICAL-INEQUALITY, MISOGYNY-NON-SEXUAL-V...",DIRECT
349,211204,en,A new core memory!,YES,DIRECT,"[MISOGYNY-NON-SEXUAL-VIOLENCE, OBJECTIFICATION...",DIRECT


## Set the seed

In [ ]:
def set_seed(seed=1234):
    """
    Sets the seed to make everything deterministic, for reproducibility of experiments
    Parameters:
    seed: the number to set the seed to
    Return: None
    """
    # Random seed
    random.seed(seed)
    # Numpy seed
    np.random.seed(seed)
    # Torch seed
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    # os seed
    os.environ['PYTHONHASHSEED'] = str(seed)

## Dataset class

In [ ]:
class SexismDataset(Dataset):
    def __init__(self, texts, labels, ids, tokenizer, max_len=128, pad="max_length", trunc=True,rt='pt'):
        self.texts = texts.tolist()
        self.labels = labels
        self.ids = ids
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pad = pad
        self.trunc = trunc
        self.rt = rt

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,padding=self.pad, truncation=self.trunc,
            return_tensors=self.rt
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
            'id': torch.tensor(self.ids[idx], dtype=torch.long)
        }
        

class SexismDatasetMulti(Dataset):
  def __init__(self, texts, labels, ids, tokenizer, max_len=128, pad="max_length", trunc=True, rt='pt'):
      self.texts = texts.tolist()
      self.labels = labels
      self.ids = ids
      self.tokenizer = tokenizer
      self.max_len = max_len
      self.pad = pad
      self.trunc = trunc
      self.rt = rt
  def __len__(self):
      return len(self.texts)
  def __getitem__(self, idx):
      text = str(self.texts[idx])
      inputs = self.tokenizer(
          text,
          add_special_tokens=True,
          max_length=self.max_len,
          padding=self.pad,
          truncation=self.trunc,
          return_tensors=self.rt
      )
      return {
          "input_ids": inputs["input_ids"].flatten(),
          "attention_mask": inputs["attention_mask"].flatten(),
          "labels": torch.tensor(self.labels[idx], dtype=torch.float),
          "id": torch.tensor(self.ids[idx], dtype=torch.long),
      }

## Metrics for subtasks 2.1 and 2.2

In [ ]:
def compute_metrics_1_2(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0
    )
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

## Metrics for subtasks 2.3

In [ ]:
def compute_metrics_3(pred, lencoder):
    labels = pred.label_ids
    preds = torch.sigmoid(torch.tensor(pred.predictions)).numpy()
    preds_binary = (preds >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds_binary, average=None, zero_division=0
    )
    acc = accuracy_score(labels, preds_binary)
    # UCM must be used for the competition
    #icm= ICMWrapper(lencoder.inverse_transform(preds_binary), lencoder.inverse_transform(labels), multi=True)
    # Macro averages
    precision_macro = np.mean(precision)
    recall_macro = np.mean(recall)
    f1_macro = np.mean(f1)
    return {
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        #'ICM': icm
    }

## Discriminative fine-tuning pipeline for subtasks 2.1 and 2.2 (binary classification tasks)

In [ ]:
def sexism_classification_pipeline_task1_2(trainLabelDf, devLabelDf, task_name, mapping, testInfo=None, model_name='roberta-base', nlabels=2, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc= LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    # Prepare datasets
    train_dataset = SexismDataset(trainLabelDf["text"], [mapping[l] for l in trainLabelDf["label"]], [int(x) for x in trainLabelDf["id_EXIST"]], tokenizer)
    val_dataset = SexismDataset(devLabelDf["text"], [mapping[l] for l in devLabelDf["label"]], [int(x) for x in devLabelDf["id_EXIST"]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1"),
        disable_tqdm=True
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_1_2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    # If there is a test dataset. Only for submissions to the shared-task
    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo["text"], [0] * len(testInfo["text"]),  [int(x) for x in testInfo["id_EXIST"]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame. Adapt it to the format of the competition.
        submission_df = pd.DataFrame({
            'id': testInfo["id_EXIST"],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2026"]*len(predicted_labels)
        })
        submission_df.to_csv(task_name + '.csv', index=False)
        print("Prediction for " + task_name + " completed. Results saved to " + task_name + ".csv")
        return model, submission_df
    return model, eval_results



## Discriminative fine-tuning pipeline for subtask 2.3 (multi-class multi-label classification task)

In [ ]:
def sexism_classification_pipeline_task3(trainMultiDf, devMultiDf, task_name, testInfo=None, model_name='roberta-base', nlabels=5, ptype="multi_label_classification", **args):
    # Model and Tokenizer
    labelEnc= MultiLabelBinarizer(classes=LABELS_23)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype)

    # Prepare datasets
    train_dataset = SexismDatasetMulti(trainMultiDf["text"], labelEnc.fit_transform(trainMultiDf["labels_23"]), [int(x) for x in trainMultiDf["id_EXIST"]], tokenizer)
    val_dataset = SexismDatasetMulti(devMultiDf["text"], labelEnc.transform(devMultiDf["labels_23"]), [int(x) for x in devMultiDf["id_EXIST"]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1_macro"),
        disable_tqdm=True
    )

    # Initialize Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=partial(compute_metrics_3, lencoder=labelEnc),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    if testInfo is not None:
      # Prepare test dataset for prediction
      test_dataset = SexismDatasetMulti(testInfo["text"], [[0, 0, 0, 0, 0]] * len(testInfo["text"]), [int(x) for x in testInfo["id_EXIST"]], tokenizer)

      # Predict test set labels
      predictions = trainer.predict(test_dataset)
      predicted_probs = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
      predicted_labels = (predicted_probs >= 0.5).astype(int)

      # Create submission DataFrame
      submission_df = pd.DataFrame({
          'id': testInfo["id_EXIST"],
          'label': labelEnc.inverse_transform(predicted_labels),
          "test_case": ["EXIST2026"] * len(predicted_labels)

      })
      submission_df.to_csv(task_name + '.csv', index=False)
      print("Prediction for " + task_name + " completed. Results saved to " + task_name + ".csv")
      return model, submission_df
    return model, eval_results

## Addressing english subtask 1

In [ ]:
# Usage Example
set_seed()

# Choose backbone for this quick run: "roberta" or "bert"
EN_BACKBONES = {
    "roberta": "cardiffnlp/twitter-roberta-base-2022-154m",
    "bert": "bert-base-uncased",
}
modelname = EN_BACKBONES["roberta"]

# training params
params = {
    "num_train_epochs": 10
    # "early_stopping_patience": 3
}
time0 = time.time()
m, res_en1 = sexism_classification_pipeline_task1_2(train21_en, dev21_en, "exist_task1_eng", MAPPING_21, None, modelname, 2, "single_label_classification", **params )
time_en1 = time.time()-time0

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.6953, 'grad_norm': 1.0681575536727905, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.11627906976744186}
{'loss': 0.697, 'grad_norm': 0.8654901385307312, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.23255813953488372}
{'loss': 0.6887, 'grad_norm': 2.1365349292755127, 'learning_rate': 3e-06, 'epoch': 0.3488372093023256}
{'loss': 0.6886, 'grad_norm': 6.348840236663818, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.46511627906976744}
{'loss': 0.6939, 'grad_norm': 0.9449529647827148, 'learning_rate': 5e-06, 'epoch': 0.5813953488372093}
{'loss': 0.6872, 'grad_norm': 2.596917152404785, 'learning_rate': 6e-06, 'epoch': 0.6976744186046512}
{'loss': 0.6754, 'grad_norm': 2.9440104961395264, 'learning_rate': 7.000000000000001e-06, 'epoch': 0.813953488372093}
{'loss': 0.6832, 'grad_norm': 2.665419340133667, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.9302325581395349}
{'eval_loss': 0.6767351031303406, 'eval_accuracy': 0.5659824046920822, 'eval_f1': 0.722846441947

## LoRA pipeline for subtasks 1 and 2 (binary classification tasks)

In [ ]:
######################################CHANGE###############################################
from peft import LoraConfig, get_peft_model, TaskType
###########################################################################################
def sexism_classification_pipeline_task1_2_LoRA(trainLabelDf, devLabelDf, task_name, mapping, testInfo=None, model_name='roberta-base', nlabels=2, ptype="single_label_classification", **args):
    # Model and Tokenizer
    labelEnc = LabelEncoder()
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    ######################################CHANGE###############################################
    # Configure LoRA
    lora_config = LoraConfig(
    task_type= args.get("task_type", TaskType.SEQ_CLS),
    target_modules= args.get("target_modules", ["query", "value"]),
    r= args.get("rank", 64),  # Rank of LoRA adaptation
    lora_alpha=args.get("lora_alpha", 32),  # Scaling factor
    lora_dropout=args.get("lora_dropout", 0.1),
    bias=args.get("bias", "none")
)
    ###########################################################################################

    ######################################CHANGE###############################################
    # Prepare LoRA model
    peft_model = get_peft_model(model, lora_config)

    ###########################################################################################
    # Prepare datasets
    train_dataset = SexismDataset(trainLabelDf["text"], [mapping[l] for l in trainLabelDf["label"]], [int(x) for x in trainLabelDf["id_EXIST"]], tokenizer)
    val_dataset = SexismDataset(devLabelDf["text"], [mapping[l] for l in devLabelDf["label"]], [int(x) for x in devLabelDf["id_EXIST"]], tokenizer)

    # Training Arguments
    training_args = TrainingArguments(
        report_to="none", # alt: "wandb", "tensorboard" "comet_ml" "mlflow" "clearml"
        output_dir= args.get('output_dir', './results_task1_LoRA0'),
        num_train_epochs= args.get('num_train_epochs', 5),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 16),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 64),
        warmup_steps=args.get('warmup_steps', 500),
        weight_decay=args.get('weight_decay',0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 10),
        eval_strategy=args.get('eval_strategy','epoch'),
        save_strategy=args.get('save_strategy', "epoch"),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model',"f1"),
        disable_tqdm=True
    )

    # Initialize Trainer
    trainer = Trainer(
        ######################################CHANGE###############################################
        # Prepare LoRA model
        model=peft_model,
        ###########################################################################################
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics_1_2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience",3))]
    )

    # Fine-tune the model
    trainer.train()

    # Evaluate on validation set
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    ######################################CHANGE###############################################
    # Saving the new weights for the LoRA model without overwriting other experiments
    lora_model_dir = args.get("lora_model_dir", "./final_best_model_LoRA")
    merged_model_dir = args.get("merged_model_dir", "./final_best_model_mixpeft")
    trainer.save_model(lora_model_dir)
    # Notice that, in this case only the LoRA matrices are saved.
    # The weights for the classification head are not saved.
    ###########################################################################################

    ######################################CHANGE###############################################
    # Mixing the LoRA matrices with the weights of the base model used
    mixModel = peft_model.merge_and_unload()
    mixModel.save_pretrained(merged_model_dir)
    # In this case the full model is saved.
    ###########################################################################################

    # If there is a test dataset. Only for submissions to the shared-task
    if testInfo is not None:
        # Prepare test dataset for prediction
        test_dataset = SexismDataset(testInfo["text"], [0] * len(testInfo["text"]),  [int(x) for x in testInfo["id_EXIST"]],   tokenizer)

        # Predict test set labels
        predictions = trainer.predict(test_dataset)
        predicted_labels = np.argmax(predictions.predictions, axis=1)

        # Create submission DataFrame
        submission_df = pd.DataFrame({
            'id': testInfo["id_EXIST"],
            'label': labelEnc.inverse_transform(predicted_labels),
            "test_case": ["EXIST2026"]*len(predicted_labels)
        })
        submission_df.to_csv(task_name + '.csv', index=False)
        print("Prediction for " + task_name + " completed. Results saved to " + task_name + ".csv")
        return model, submission_df
    return model, eval_results

# Experimental work

In [ ]:
import gc


# Some Spanish models:
# "pysentimiento/robertuito-base-uncased"
# "pysentimiento/robertuito-hate-speech"
# "pysentimiento/robertuito-base-deacc"
# "dccuchile/bert-base-spanish-wwm-uncased"
# NOTE: "PlanTL-GOB-ES/roberta-base-bne" and "PlanTL-GOB-ES/roberta-large-bne"
# have been emptied upstream (only README/.gitattributes remain) and can no
# longer be loaded. Using the BERTIN Spanish RoBERTa as a drop-in replacement.

EN_BACKBONES = {
    "roberta": "cardiffnlp/twitter-roberta-base-2022-154m",
    "bert": "bert-base-uncased",
}
ES_BACKBONES = {
    "roberta": "bertin-project/bertin-roberta-base-spanish",
    "bert": "dccuchile/bert-base-spanish-wwm-uncased",
}

# Set the active backbone family for this experiment: "roberta" or "bert"
ACTIVE_BACKBONE = "roberta"

ENGLISH_MODEL = EN_BACKBONES[ACTIVE_BACKBONE]
SPANISH_MODEL = ES_BACKBONES[ACTIVE_BACKBONE]
OUTPUT_ROOT = Path(f"./pract_5_experiments_{ACTIVE_BACKBONE}")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def cleanup_after_training(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sexism_classification_pipeline_task3_LoRA(trainMultiDf, devMultiDf, task_name, testInfo=None, model_name='roberta-base', nlabels=5, ptype="multi_label_classification", **args):
    labelEnc = MultiLabelBinarizer(classes=LABELS_23)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=nlabels,
        problem_type=ptype
    )

    lora_config = LoraConfig(
        task_type=args.get("task_type", TaskType.SEQ_CLS),
        target_modules=args.get("target_modules", ["query", "value"]),
        r=args.get("rank", 64),
        lora_alpha=args.get("lora_alpha", 32),
        lora_dropout=args.get("lora_dropout", 0.1),
        bias=args.get("bias", "none")
    )
    peft_model = get_peft_model(model, lora_config)

    train_dataset = SexismDatasetMulti(trainMultiDf["text"], labelEnc.fit_transform(trainMultiDf["labels_23"]), [int(x) for x in trainMultiDf["id_EXIST"]], tokenizer)
    val_dataset = SexismDatasetMulti(devMultiDf["text"], labelEnc.transform(devMultiDf["labels_23"]), [int(x) for x in devMultiDf["id_EXIST"]], tokenizer)

    training_args = TrainingArguments(
        report_to="none",
        output_dir=args.get('output_dir', './results_task3_LoRA'),
        num_train_epochs=args.get('num_train_epochs', 3),
        learning_rate=args.get('learning_rate', 5e-5),
        per_device_train_batch_size=args.get('per_device_train_batch_size', 8),
        per_device_eval_batch_size=args.get('per_device_eval_batch_size', 16),
        warmup_steps=args.get('warmup_steps', 0),
        weight_decay=args.get('weight_decay', 0.01),
        logging_dir=args.get('logging_dir', './logs'),
        logging_steps=args.get('logging_steps', 50),
        eval_strategy=args.get('eval_strategy', 'epoch'),
        save_strategy=args.get('save_strategy', 'epoch'),
        save_total_limit=args.get('save_total_limit', 1),
        load_best_model_at_end=args.get('load_best_model_at_end', True),
        metric_for_best_model=args.get('metric_for_best_model', 'f1_macro'),
        disable_tqdm=True
    )

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=partial(compute_metrics_3, lencoder=labelEnc),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=args.get("early_stopping_patience", 2))]
    )

    trainer.train()
    eval_results = trainer.evaluate()
    print("Validation Results:", eval_results)

    lora_model_dir = args.get("lora_model_dir", "./final_best_model_LoRA_task3")
    merged_model_dir = args.get("merged_model_dir", "./final_best_model_mixpeft_task3")
    trainer.save_model(lora_model_dir)
    merged_model = peft_model.merge_and_unload()
    merged_model.save_pretrained(merged_model_dir)
    del merged_model

    if testInfo is not None:
        test_dataset = SexismDatasetMulti(testInfo["text"], [[0, 0, 0, 0, 0]] * len(testInfo["text"]), [int(x) for x in testInfo["id_EXIST"]], tokenizer)
        predictions = trainer.predict(test_dataset)
        predicted_probs = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
        predicted_labels = (predicted_probs >= 0.5).astype(int)
        submission_df = pd.DataFrame({
            'id': testInfo["id_EXIST"],
            'label': labelEnc.inverse_transform(predicted_labels),
            'test_case': ["EXIST2026"] * len(predicted_labels)
        })
        submission_df.to_csv(task_name + '.csv', index=False)
        print("Prediction for " + task_name + " completed. Results saved to " + task_name + ".csv")
        return None, submission_df

    return None, eval_results


COMMON_TRAIN_ARGS = {
    "num_train_epochs": 3,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "warmup_steps": 0,
    "weight_decay": 0.01,
    "logging_steps": 50,
    "save_total_limit": 1,
    "load_best_model_at_end": True,
    "early_stopping_patience": 2,
}

experiments = [
    {
        "language": "English",
        "task": "2.1",
        "method": "FT",
        "run_name": "en_task21_ft",
        "train_df": train21_en,
        "dev_df": dev21_en,
        "pipeline": sexism_classification_pipeline_task1_2,
        "mapping": MAPPING_21,
        "model_name": ENGLISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "English",
        "task": "2.1",
        "method": "LoRA",
        "run_name": "en_task21_lora",
        "train_df": train21_en,
        "dev_df": dev21_en,
        "pipeline": sexism_classification_pipeline_task1_2_LoRA,
        "mapping": MAPPING_21,
        "model_name": ENGLISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "English",
        "task": "2.2",
        "method": "FT",
        "run_name": "en_task22_ft",
        "train_df": train22_en,
        "dev_df": dev22_en,
        "pipeline": sexism_classification_pipeline_task1_2,
        "mapping": MAPPING_22,
        "model_name": ENGLISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "English",
        "task": "2.2",
        "method": "LoRA",
        "run_name": "en_task22_lora",
        "train_df": train22_en,
        "dev_df": dev22_en,
        "pipeline": sexism_classification_pipeline_task1_2_LoRA,
        "mapping": MAPPING_22,
        "model_name": ENGLISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "English",
        "task": "2.3",
        "method": "FT",
        "run_name": "en_task23_ft",
        "train_df": train23_en,
        "dev_df": dev23_en,
        "pipeline": sexism_classification_pipeline_task3,
        "mapping": None,
        "model_name": ENGLISH_MODEL,
        "nlabels": 5,
        "ptype": "multi_label_classification",
    },
    {
        "language": "English",
        "task": "2.3",
        "method": "LoRA",
        "run_name": "en_task23_lora",
        "train_df": train23_en,
        "dev_df": dev23_en,
        "pipeline": sexism_classification_pipeline_task3_LoRA,
        "mapping": None,
        "model_name": ENGLISH_MODEL,
        "nlabels": 5,
        "ptype": "multi_label_classification",
    },
    {
        "language": "Spanish",
        "task": "2.1",
        "method": "FT",
        "run_name": "es_task21_ft",
        "train_df": train21_es,
        "dev_df": dev21_es,
        "pipeline": sexism_classification_pipeline_task1_2,
        "mapping": MAPPING_21,
        "model_name": SPANISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "Spanish",
        "task": "2.1",
        "method": "LoRA",
        "run_name": "es_task21_lora",
        "train_df": train21_es,
        "dev_df": dev21_es,
        "pipeline": sexism_classification_pipeline_task1_2_LoRA,
        "mapping": MAPPING_21,
        "model_name": SPANISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "Spanish",
        "task": "2.2",
        "method": "FT",
        "run_name": "es_task22_ft",
        "train_df": train22_es,
        "dev_df": dev22_es,
        "pipeline": sexism_classification_pipeline_task1_2,
        "mapping": MAPPING_22,
        "model_name": SPANISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "Spanish",
        "task": "2.2",
        "method": "LoRA",
        "run_name": "es_task22_lora",
        "train_df": train22_es,
        "dev_df": dev22_es,
        "pipeline": sexism_classification_pipeline_task1_2_LoRA,
        "mapping": MAPPING_22,
        "model_name": SPANISH_MODEL,
        "nlabels": 2,
        "ptype": "single_label_classification",
    },
    {
        "language": "Spanish",
        "task": "2.3",
        "method": "FT",
        "run_name": "es_task23_ft",
        "train_df": train23_es,
        "dev_df": dev23_es,
        "pipeline": sexism_classification_pipeline_task3,
        "mapping": None,
        "model_name": SPANISH_MODEL,
        "nlabels": 5,
        "ptype": "multi_label_classification",
    },
    {
        "language": "Spanish",
        "task": "2.3",
        "method": "LoRA",
        "run_name": "es_task23_lora",
        "train_df": train23_es,
        "dev_df": dev23_es,
        "pipeline": sexism_classification_pipeline_task3_LoRA,
        "mapping": None,
        "model_name": SPANISH_MODEL,
        "nlabels": 5,
        "ptype": "multi_label_classification",
    },
]

results = []

for experiment in experiments:
    run_dir = OUTPUT_ROOT / experiment["run_name"]
    run_dir.mkdir(parents=True, exist_ok=True)

    run_args = dict(COMMON_TRAIN_ARGS)
    run_args.update({
        "output_dir": str(run_dir / "checkpoints"),
        "logging_dir": str(run_dir / "logs"),
    })

    if experiment["method"] == "LoRA":
        run_args.update({
            "lora_model_dir": str(run_dir / "lora_adapter"),
            "merged_model_dir": str(run_dir / "merged_model"),
        })

    print(f"Running {experiment['run_name']} with {experiment['model_name']}")
    time0 = time.time()
    set_seed()

    if experiment["mapping"] is None:
        model, eval_results = experiment["pipeline"](
            experiment["train_df"],
            experiment["dev_df"],
            experiment["run_name"],
            None,
            experiment["model_name"],
            experiment["nlabels"],
            experiment["ptype"],
            **run_args,
        )
    else:
        model, eval_results = experiment["pipeline"](
            experiment["train_df"],
            experiment["dev_df"],
            experiment["run_name"],
            experiment["mapping"],
            None,
            experiment["model_name"],
            experiment["nlabels"],
            experiment["ptype"],
            **run_args,
        )

    elapsed = time.time() - time0
    row = {
        "language": experiment["language"],
        "task": experiment["task"],
        "method": experiment["method"],
        "run_name": experiment["run_name"],
        "model_name": experiment["model_name"],
        "training_time_sec": elapsed,
        "output_dir": str(run_dir),
    }
    row.update(eval_results)
    row["primary_metric"] = eval_results.get("eval_f1", eval_results.get("eval_f1_macro"))
    results.append(row)

    cleanup_after_training(model)

results_df = pd.DataFrame(results).sort_values(["language", "task", "method"]).reset_index(drop=True)
results_df

Running en_task21_ft with cardiffnlp/twitter-roberta-base-2022-154m


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.6929, 'grad_norm': 17.274961471557617, 'learning_rate': 4.512670565302144e-05, 'epoch': 0.29239766081871343}
{'loss': 0.6803, 'grad_norm': 7.518261909484863, 'learning_rate': 4.025341130604289e-05, 'epoch': 0.5847953216374269}
{'loss': 0.6376, 'grad_norm': 4.204442024230957, 'learning_rate': 3.538011695906433e-05, 'epoch': 0.8771929824561403}
{'eval_loss': 0.679513692855835, 'eval_accuracy': 0.6539589442815249, 'eval_f1': 0.719047619047619, 'eval_precision': 0.6651982378854625, 'eval_recall': 0.7823834196891192, 'eval_runtime': 0.5663, 'eval_samples_per_second': 602.118, 'eval_steps_per_second': 38.846, 'epoch': 1.0}
{'loss': 0.6774, 'grad_norm': 5.981500148773193, 'learning_rate': 3.050682261208577e-05, 'epoch': 1.1695906432748537}
{'loss': 0.6381, 'grad_norm': 4.853490829467773, 'learning_rate': 2.5633528265107216e-05, 'epoch': 1.4619883040935673}
{'loss': 0.6177, 'grad_norm': 4.326906681060791, 'learning_rate': 2.0760233918128656e-05, 'epoch': 1.7543859649122808}
{'eval_l

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-roberta-base-2022-154m and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.6934, 'grad_norm': 1.4521363973617554, 'learning_rate': 4.512670565302144e-05, 'epoch': 0.29239766081871343}
{'loss': 0.6884, 'grad_norm': 2.383704900741577, 'learning_rate': 4.025341130604289e-05, 'epoch': 0.5847953216374269}
{'loss': 0.6746, 'grad_norm': 1.3220813274383545, 'learning_rate': 3.538011695906433e-05, 'epoch': 0.8771929824561403}
{'eval_loss': 0.6644462943077087, 'eval_accuracy': 0.5659824046920822, 'eval_f1': 0.7228464419475655, 'eval_precision': 0.5659824046920822, 'eval_recall': 1.0, 'eval_runtime': 0.575, 'eval_samples_per_second': 593.084, 'eval_steps_per_second': 38.263, 'epoch': 1.0}
{'loss': 0.6697, 'grad_norm': 0.9539312720298767, 'learning_rate': 3.050682261208577e-05, 'epoch': 1.1695906432748537}
{'loss': 0.6535, 'grad_norm': 1.2933597564697266, 'learning_rate': 2.5633528265107216e-05, 'epoch': 1.4619883040935673}
{'loss': 0.6177, 'grad_norm': 2.3249292373657227, 'learning_rate': 2.0760233918128656e-05, 'epoch': 1.7543859649122808}
{'eval_loss': 0.59

...

{'loss': 0.5069, 'grad_norm': 1.792358636856079, 'learning_rate': 3.611111111111111e-05, 'epoch': 0.8333333333333334}
{'eval_loss': 0.40550854802131653, 'eval_accuracy': 0.8583333333333333, 'eval_f1': 0.9237668161434978, 'eval_precision': 0.8583333333333333, 'eval_recall': 1.0, 'eval_runtime': 0.2111, 'eval_samples_per_second': 568.578, 'eval_steps_per_second': 37.905, 'epoch': 1.0}
{'loss': 0.3764, 'grad_norm': 1.0205037593841553, 'learning_rate': 2.2222222222222223e-05, 'epoch': 1.6666666666666665}
{'eval_loss': 0.41050708293914795, 'eval_accuracy': 0.8583333333333333, 'eval_f1': 0.9237668161434978, 'eval_precision': 0.8583333333333333, 'eval_recall': 1.0, 'eval_runtime': 0.204, 'eval_samples_per_second': 588.341, 'eval_steps_per_second': 39.223, 'epoch': 2.0}
{'loss': 0.3637, 'grad_norm': 0.7856732606887817, 'learning_rate': 8.333333333333334e-06, 'epoch': 2.5}
{'eval_loss': 0.4117749333381653, 'eval_accuracy': 0.8583333333333333, 'eval_f1': 0.9237668161434978, 'eval_precision': 0.8

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at bertin-project/bertin-roberta-base-spanish and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.658, 'grad_norm': 0.738968014717102, 'learning_rate': 4.198717948717949e-05, 'epoch': 0.4807692307692308}
{'loss': 0.635, 'grad_norm': 0.6593613624572754, 'learning_rate': 3.397435897435898e-05, 'epoch': 0.9615384615384616}
{'eval_loss': 0.6383020877838135, 'eval_precision_macro': 0.4230769230769231, 'eval_recall_macro': 0.6, 'eval_f1_macro': 0.4957813316843701, 'eval_runtime': 0.3636, 'eval_samples_per_second': 572.133, 'eval_steps_per_second': 35.758, 'epoch': 1.0}
{'loss': 0.6429, 'grad_norm': 0.3864787817001343, 'learning_rate': 2.5961538461538464e-05, 'epoch': 1.4423076923076923}
{'loss': 0.6323, 'grad_norm': 0.3827328681945801, 'learning_rate': 1.794871794871795e-05, 'epoch': 1.9230769230769231}
{'eval_loss': 0.6397392749786377, 'eval_precision_macro': 0.525, 'eval_recall_macro': 0.8, 'eval_f1_macro': 0.630813178818128, 'eval_runtime': 0.3552, 'eval_samples_per_second': 585.576, 'eval_steps_per_second': 36.599, 'epoch': 2.0}
{'loss': 0.6405, 'grad_norm': 0.301183104515

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at bertin-project/bertin-roberta-base-spanish and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'loss': 0.6661, 'grad_norm': 0.4825531542301178, 'learning_rate': 4.198717948717949e-05, 'epoch': 0.4807692307692308}
{'loss': 0.6327, 'grad_norm': 0.5409879684448242, 'learning_rate': 3.397435897435898e-05, 'epoch': 0.9615384615384616}
{'eval_loss': 0.6371160745620728, 'eval_precision_macro': 0.5264102564102564, 'eval_recall_macro': 0.7754716981132075, 'eval_f1_macro': 0.6258512617543002, 'eval_runtime': 0.3654, 'eval_samples_per_second': 569.183, 'eval_steps_per_second': 35.574, 'epoch': 1.0}
{'loss': 0.6368, 'grad_norm': 0.3411184251308441, 'learning_rate': 2.5961538461538464e-05, 'epoch': 1.4423076923076923}
{'loss': 0.6281, 'grad_norm': 0.32481297850608826, 'learning_rate': 1.794871794871795e-05, 'epoch': 1.9230769230769231}
{'eval_loss': 0.6363469362258911, 'eval_precision_macro': 0.525, 'eval_recall_macro': 0.8, 'eval_f1_macro': 0.630813178818128, 'eval_runtime': 0.3503, 'eval_samples_per_second': 593.715, 'eval_steps_per_second': 37.107, 'epoch': 2.0}
{'loss': 0.6338, 'grad_no

# Show results

In [ ]:
if "results_df" not in globals() or results_df.empty:
    raise ValueError("Run the experimental work cell first to generate results_df.")

summary_columns = [
    "language",
    "task",
    "method",
    "model_name",
    "primary_metric",
    "eval_accuracy",
    "eval_f1",
    "eval_precision",
    "eval_recall",
    "eval_f1_macro",
    "eval_precision_macro",
    "eval_recall_macro",
    "training_time_sec",
]

display(results_df[[col for col in summary_columns if col in results_df.columns]])

binary_results = results_df[results_df["task"].isin(["2.1", "2.2"])].copy()
if not binary_results.empty:
    display(
        binary_results.pivot_table(
            index=["language", "task"],
            columns="method",
            values="eval_f1",
            aggfunc="first",
        ).sort_index()
    )

multilabel_results = results_df[results_df["task"] == "2.3"].copy()
if not multilabel_results.empty:
    display(
        multilabel_results.pivot_table(
            index=["language", "task"],
            columns="method",
            values="eval_f1_macro",
            aggfunc="first",
        ).sort_index()
    )